# VisionX — اختبار الـ pipeline

يشغّل نفس مسار `main.py`: كشف سيارة ← براند/موديل ← لوحة ← تحسين ← Qwen.

- **الافتراضي: تشغيل Qwen 2.5-VL** لقراءة اللوحة حتى تُطابق قاعدة البيانات بالرقم. أول تشغيل قد يحمّل الموديل من Hugging Face. للتجربة السريعة فقط: `SKIP_QWEN = True`.
- موديل البراند: يُختار تلقائياً أول ملف موجود بين `best.pt` / `brand_model.pt` / `weights.pt` في `models/car_brand_model_detector/`.
- الصور تُنسَخ من سطح المكتب `test_images` إلى مجلد المشروع عند التشغيل.
- للمقارنة مع قاعدة البيانات: `python db_cli.py link --image ...` مسبقاً ثم فعّل `COMPARE_DB`.


In [ ]:
%pip install -q ultralytics opencv-python pillow matplotlib pandas transformers accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TEST_IMAGES_DIR = PROJECT_ROOT / "test_images"
DESKTOP_TEST_IMAGES = Path(r"C:\Users\SWEMa\Desktop\test_images")

TEST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
if DESKTOP_TEST_IMAGES.is_dir():
    copied = []
    for f in DESKTOP_TEST_IMAGES.iterdir():
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            shutil.copy2(f, TEST_IMAGES_DIR / f.name)
            copied.append(f.name)
    print("Synced from Desktop:", copied if copied else "empty")
else:
    print("Desktop folder not found:", DESKTOP_TEST_IMAGES)

test_files = sorted(
    {p for pat in ("*.jpg", "*.jpeg", "*.png", "*.webp") for p in TEST_IMAGES_DIR.glob(pat)}
)
print("Images in project test_images/:", [p.name for p in test_files])


Synced from Desktop: ['WhatsApp Image 2026-04-04 at 10.50.30 PM.jpeg', 'WhatsApp Image 2026-04-04 at 4.10.34 PM.jpeg']
Images in project test_images/: ['WhatsApp Image 2026-04-04 at 10.50.30 PM.jpeg', 'WhatsApp Image 2026-04-04 at 4.10.34 PM.jpeg']


In [4]:
def show_img(img, title="", figsize=(10, 5)):
    if img is None:
        print("(no image)", title)
        return
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


In [5]:
from IPython.display import display

from main import run_pipeline
from utils.config import DEFAULT_DB_PATH
from utils.database import (
    compare_prediction_to_ground_truth,
    get_expected_for_basename,
    get_ground_truth_by_plate_final,
)

# False = يشغّل Qwen 2.5-VL لقراءة اللوحة (مطلوب لمطابقة DB بالرقم). True = تخطٍ للتجربة السريعة فقط
SKIP_QWEN = False
SKIP_BRAND = False
# True = يتخطى موديل اللون (Keras) إذا ما ثبّت tensorflow أو ملف .keras
SKIP_COLOR = False
COMPARE_DB = True

rows = []
summaries = {}

for img_path in test_files:
    out_dir = PROJECT_ROOT / "outputs" / "notebook_test" / img_path.stem
    summary = run_pipeline(
        img_path,
        out_dir,
        skip_brand=SKIP_BRAND,
        skip_color=SKIP_COLOR,
        skip_qwen=SKIP_QWEN,
    )
    summaries[img_path.name] = summary

    pr = summary.get("plate_read") or {}
    bm = summary.get("brand_model") or {}
    vc = summary.get("vehicle_color") or {}
    row = {
        "image": img_path.name,
        "vehicle_ok": summary.get("vehicle") is not None,
        "plate_ok": summary.get("plate") is not None,
        "digits": pr.get("digits"),
        "letters": pr.get("letters"),
        "final": pr.get("final"),
        "brand": bm.get("brand"),
        "model": bm.get("model"),
        "pred_color": vc.get("color"),
        "overview": str(out_dir / "00_overview.jpg"),
    }

    if COMPARE_DB:
        exp = get_expected_for_basename(img_path.name, db_path=DEFAULT_DB_PATH)
        if exp is None and pr.get("final"):
            exp = get_ground_truth_by_plate_final(pr["final"], db_path=DEFAULT_DB_PATH)
        if exp is None:
            row["db_plate_final_match"] = None
            row["db_note"] = "no row (import Excel or db_cli link)"
        else:
            comp = compare_prediction_to_ground_truth(exp, summary)
            row["db_plate_final_match"] = comp["plate_final_match"]
            row["db_all_plate"] = comp["all_plate_fields_match"]
            row["db_brand_match"] = comp["brand_match"]
            row["db_color_match"] = comp.get("color_match")
            row["registry_ok"] = comp.get("registry_consistent")
            row["expected_final"] = exp.get("plate_final")
            row["expected_color"] = exp.get("expected_color")
            summary["db_comparison"] = comp
        with open(out_dir / "summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

    rows.append(row)

df = pd.DataFrame(rows)
display(df)



============ VEHICLE ============

0: 640x512 1 car, 209.3ms
Speed: 31.3ms preprocess, 209.3ms inference, 15.4ms postprocess per image at shape (1, 3, 640, 512)
bbox: [131, 159, 1178, 1283] conf: 0.8695525527000427

============ BRAND / MODEL ============
Brand/model classifier not found: C:\Users\SWEMa\Desktop\computer-vision-project-visionx\models\car_brand_model_detector\best.pt. Export best.pt from Colab into models/car_brand_model_detector/ or set VISIONX_BRAND_MODEL.

============ PLATE ============

0: 960x896 1 license_plate, 793.1ms
Speed: 10.6ms preprocess, 793.1ms inference, 1.5ms postprocess per image at shape (1, 3, 960, 896)
bbox (in ROI): [436, 934, 648, 1041] conf: 0.8818086981773376

============ ENHANCE + READ ============


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# عرض overview لكل صورة
for img_path in test_files:
    p = PROJECT_ROOT / "outputs" / "notebook_test" / img_path.stem / "00_overview.jpg"
    if p.is_file():
        show_img(cv2.imread(str(p)), title=img_path.name)
    else:
        print("Missing overview:", p)
